# Export Earth Engine images and tables with pysepal-api

Run this notebook inside your SEPAL workspace with a version of `pysepal-api[ee]` that includes `ee_exports` already installed. No fork or local source checkout is required. The SEPAL backend must support the `ee.export` operation, and SEPAL API authentication and Earth Engine credentials must already be configured.

Run the setup and sample-object cells first, then choose the synchronous or asynchronous examples. Each export cell submits a real task. Results go into a unique directory under `~/module_results/ee_export_smoke_test/`.

Destinations use the `EEExportDestination` enum: `SEPAL` for workspace files, `DRIVE` for Google Drive, and `GEE` for Earth Engine assets.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import ee
from pysepal_api import AsyncSepalClient, EEExportDestination, SepalClient

sepal = SepalClient(timeout=60)
run_name = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
output_dir = f"module_results/ee_export_smoke_test/{run_name}"
print("SEPAL:", sepal.base_url)
print("Workspace entries:", len(sepal.files.list(".")))
print("Output:", Path.home() / output_dir)


## Make tiny Earth Engine objects

`ee.Initialize()` uses the Earth Engine credentials and project configured in your SEPAL workspace. The sample image contains a constant value of 7 over a small rectangle near Rome.


In [ ]:
ee.Initialize()
region = ee.Geometry.Rectangle([12.49, 41.89, 12.50, 41.90])
image = ee.Image.constant(7).rename("test_value").toInt16().clip(region)
plots = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([12.495, 41.895]), {"id": 1, "label": "export_smoke_test"})
])
print("Earth Engine objects ready")


## Export an image to the SEPAL workspace

Running the next cell submits an actual export. You can also follow it in SEPAL's **Tasks** panel.


In [ ]:
image_task = sepal.ee_exports.submit_image(
    image=image,
    region=region,
    destination=EEExportDestination.SEPAL,
    name=f"image_smoke_{run_name}",
    scale=100,
    crs="EPSG:4326",
    workspace_path=f"{output_dir}/image",
    filename_prefix="constant_seven",
)
print("Image task:", image_task.id, image_task.state.value)


In [ ]:
image_result = sepal.tasks.wait(image_task.id, poll_interval=3, timeout=600)
print("Image export:", image_result.state.value)
image_files = sorted((Path.home() / output_dir / "image").glob("*"))
for path in image_files:
    print(path.name, path.stat().st_size, "bytes")
assert any(path.suffix == ".tif" for path in image_files), "Expected at least one GeoTIFF"


## Export a table to the SEPAL workspace

This tests the same route with a FeatureCollection. CSV exports follow the existing SEPAL exporter behavior, which includes a `.geo` geometry column.


In [ ]:
table_task = sepal.ee_exports.submit_table(
    collection=plots,
    destination=EEExportDestination.SEPAL,
    name=f"table_smoke_{run_name}",
    workspace_path=f"{output_dir}/table",
    filename_prefix="test_plots",
    file_format="CSV",
    selectors=["id", "label"],
)
print("Table task:", table_task.id, table_task.state.value)


In [ ]:
table_result = sepal.tasks.wait(table_task.id, poll_interval=3, timeout=600)
print("Table export:", table_result.state.value)
table_files = sorted((Path.home() / output_dir / "table").glob("*.csv"))
assert table_files, "Expected a CSV file"
for path in table_files:
    print(path.name)
    print(path.read_text())


## Asynchronous exports

These two optional examples reuse the setup and sample objects above; the synchronous exports do not need to be run first. Each submits an additional export to its own directory. Use notebook-level `await` to submit and wait without blocking the event loop. `async with` closes the client when the block exits; closing it does not cancel an export still running on the server.


### Export an image asynchronously


In [ ]:
async with AsyncSepalClient(timeout=60) as async_sepal:
    async_image_task = await async_sepal.ee_exports.submit_image(
        image=image,
        region=region,
        destination=EEExportDestination.SEPAL,
        name=f"async_image_smoke_{run_name}",
        scale=100,
        crs="EPSG:4326",
        workspace_path=f"{output_dir}/async_image",
        filename_prefix="constant_seven",
    )
    print("Image task:", async_image_task.id, async_image_task.state.value)
    async_image_result = await async_sepal.tasks.wait(
        async_image_task.id, poll_interval=3, timeout=600
    )
    print("Image export:", async_image_result.state.value)


### Export a table asynchronously


In [ ]:
async with AsyncSepalClient(timeout=60) as async_sepal:
    async_table_task = await async_sepal.ee_exports.submit_table(
        collection=plots,
        destination=EEExportDestination.SEPAL,
        name=f"async_table_smoke_{run_name}",
        workspace_path=f"{output_dir}/async_table",
        filename_prefix="test_plots",
        file_format="CSV",
        selectors=["id", "label"],
    )
    print("Table task:", async_table_task.id, async_table_task.state.value)
    async_table_result = await async_sepal.tasks.wait(
        async_table_task.id, poll_interval=3, timeout=600
    )
    print("Table export:", async_table_result.state.value)


## Inspect or cancel a task

If a wait times out, the export may still be running. Inspect its details with the next cell, or uncomment the cancellation line for the task you want to stop. Stopping a task cancels its Earth Engine operation and cleans temporary storage after submission. The cell below inspects the synchronous examples; for an async example, use `async_image_task` or `async_table_task` instead.

To try other destinations, use `destination=EEExportDestination.DRIVE` with `drive_folder="pysepal_api_smoke_test"`, or `destination=EEExportDestination.GEE` with your own `asset_id`. Those destinations require a linked Google account. Table `max_vertices` is currently supported only for Drive exports; ImageCollection assets use SEPAL's tiled recipe workflow.


In [ ]:
for task in (image_task, table_task):
    details = sepal.tasks.get(task.id, details=True)
    print(details.id, details.state.value, details.status_description)

# sepal.tasks.cancel(image_task.id)
# sepal.tasks.cancel(table_task.id)


In [ ]:
sepal.close()
print("Client closed. Exported files remain in", Path.home() / output_dir)
